# 14 — Preparar a avaliação temporal da referência

Autor: Wanderson Ferreira. Preparação e atualização: 2026-10-09.

**HGB = Histogram-based Gradient Boosting**, referência congelada e treinada em
abril. Este notebook registra o desenho fixado antes do acesso a 02–15/09;
suas células consultam metadados e um exemplo sintético, sem avaliar o modelo.

**Estado atual:** o executor da [etapa 15](15_reference_assessment_execution.ipynb)
foi executado e verificado pelo autor; diagnóstico e reamostragem nativa foram
revisados. O [escopo exploratório está concluído](../../docs/modeling/EVALUATION_PROTOCOL.md#resultado-nativo-da-reamostragem-e-fechamento).
02–15/09 foi consumida; 16–30/09 continua reservada para replay.


## Pergunta e alternativas

Como a referência de abril ordena as transações e prioriza clientes de 02–15/09?
Optuna conservou a referência: nenhum dos 20 trials passou o gate. Não escolhemos
retroativamente o maior AP como candidato confirmatório, nem reutilizamos o teste
de maio como confirmação independente de hipóteses posteriores.

Também não adotamos bootstrap de transações independentes: clientes, terminais e
históricos se repetem. Os 14 dias de contexto das features decorrem da causalidade
e do atraso de rótulos; não justificam blocos de reamostragem ou potência. Uma
inferência sobre períodos futuros exige condições adicionais, declaradas antes
que os resultados reservados orientem novas escolhas.


## Decisão e fronteiras

O snapshot `reference_assessment_v1` vincula os recibos de congelamento,
desenvolvimento e seleção. O plano lê somente quatro JSONs e confere hashes;
`--require-committed` também exige código e inputs rastreados, sem alterações.
O plano conserva sua função de metadados. A avaliação posterior usa a autorização
e o executor próprios da etapa 15, sem mudar este snapshot.

| Janela de 2018 | Finalidade e estado atual |
| --- | --- |
| 19/08–01/09 | Contexto causal da avaliação executada, sem treino |
| 02–15/09 | Referência fixa avaliada; janela consumida e escopo exploratório concluído |
| 16–30/09 | Reserva de replay operacional; contrato de eventos ainda a implementar |

A fila usa máximo score por cliente/dia e até 100 clientes. Essa política é
retrospectiva por dia completo. Streaming precisará definir quando emitir alertas,
tratar estado causal e entregar o feedback sete dias depois no relógio dos eventos.


In [ ]:
# Apenas metadados do checkout; sem acesso a dados/modelos nativos.
from fraud_detection_mlops.modeling.experiments.reference_temporal_assessment import (
    prepare_assessment_plan,
)

plan = prepare_assessment_plan()
assert plan["current_authorization"]["native_data_reads"] is False
assert plan["model"]["loaded"] is False
{key: value["days"] for key, value in plan["windows"].items()}

## Referência condicional de fila aleatória

Uma política hipotética sorteia clientes uniformemente, sem reposição, com
sorteios novos e independentes por dia. Condicionando nas populações e rótulos
observados, calculamos capturas esperadas, intervalo central de capturas e
probabilidade de capturar pelo menos o total observado. A convolução usa
log-probabilidades para preservar caudas pequenas.

A independência pertence aos sorteios da política, não às entidades observadas.
O intervalo não é um intervalo de confiança do HGB e a cauda não testa qualidade
futura. O exemplo abaixo é inventado, sem qualquer métrica nativa do projeto.


In [ ]:
import pandas as pd

from fraud_detection_mlops.evaluation.conditional_queue_reference import (
    uniform_queue_reference,
)

toy_days = pd.DataFrame(
    [
        {
            "date": "2000-01-01",
            "customers": 5,
            "alerts": 2,
            "fraudulent_customers": 2,
            "fraudulent_customers_in_alerts": 2,
        },
        {
            "date": "2000-01-02",
            "customers": 4,
            "alerts": 2,
            "fraudulent_customers": 1,
            "fraudulent_customers_in_alerts": 1,
        },
    ]
)
uniform_queue_reference(toy_days, alert_budget=2)

## Conferências da preparação e fechamento posterior

Os 22 casos sintéticos da preparação verificaram hashes, ausência de leitura de
dados, inputs/código commitados e rejeição de acesso reservado pela CLI de plano.
O cálculo condicional foi comparado com enumeração independente de filas possíveis
em populações pequenas, inclusive clientes recorrentes e uma cauda extrema em log10.

Naquela preparação não foram lidos dados nativos de setembro nem carregado o
modelo do autor. Os fits/runs dos testes usaram fixtures temporárias. O relato
de Optuna foi vinculado sem substituir verificação do banco nativo. Esses limites
históricos permanecem; não descrevem uma avaliação ainda pendente.

Depois, o autor executou a referência de abril em 02–15/09 e forneceu os relatórios
de avaliação, dependência e reamostragem. A grade de blocos 2/3/4/7 é exploratória,
escolhida após o diagnóstico, e não altera o protocolo prévio. Não demonstra
cobertura de generalização ou superioridade de candidato.

A próxima ação é exportar a referência existente e conferir Docker de laboratório;
Prefect e replay seguem na ordem do [mural](../../docs/project/ROADMAP.md#próxima-entrega-concreta).
Resultados detalhados e limites têm uma referência principal:
[fechamento](../../docs/modeling/EVALUATION_PROTOCOL.md#resultado-nativo-da-reamostragem-e-fechamento).
[Recibo da preparação](../../references/evidence/reference_assessment_preparation_2026-10-09.json)
e [recibo da reamostragem nativa](../../references/evidence/reference_uncertainty_author_validation_2026-10-09.json).
